# 05 — Dataset do paper e baseline

Este notebook abre a trilha de replicação fiel do artigo *Interpretability in the Wild* (arXiv:2211.00593).

Objetivos:

1. construir pares determinísticos $p_{IOI}$/$p_{ABC}$ a partir dos 15 templates históricos BABA e das variantes ABBA;
2. verificar que os nomes e substantivos usados ocupam um token do GPT-2 e que `IO`, `S1`, `S2`, `S1+1` e `END` apontam para os lugares esperados;
3. medir no GPT-2 Small a diferença de logits IO−S, a acurácia IO>S e a probabilidade absoluta de IO;
4. comparar um lote de desenvolvimento viável com os valores publicados em 100.000 exemplos, sem exigir igualdade bit a bit.

O dataset usa `prepend_bos=False`. O token final (a resposta IO) é mantido no texto histórico, mas todas as métricas usam os logits da posição semântica `END`, imediatamente anterior à resposta.

In [1]:
import json
from importlib.metadata import version
from pathlib import Path

import torch

from ioi_replication.paper import (
    build_paired_dataset,
    io_accuracy,
    io_probability,
    logit_difference,
)
from ioi_replication.runtime import load_gpt2_small, select_device

SEED = 42
N_DEV = 256
BATCH_SIZE = 32
PATTERN = "mixed"
PAPER_TARGETS = {
    "mean_logit_difference": 3.56,
    "io_accuracy": 0.993,
    "mean_io_probability": 0.49,
}

torch.manual_seed(SEED)
device = select_device()
print(f"Dispositivo: {device}; seed: {SEED}; lote dev: {N_DEV}")

Dispositivo: cuda; seed: 42; lote dev: 256


## Modelo e pares $p_{IOI}$/$p_{ABC}$

Usamos o GPT-2 Small fornecido pelo TransformerLens 4.3 em modo de compatibilidade. A implementação do dataset não importa `transformer_lens.evals.IOIDataset`: ela porta os templates e vocabulários históricos diretamente.

Em cada par, $p_{ABC}$ troca apenas `S2` por um terceiro nome single-token. Assim, template, lugar, objeto, comprimento e posições ficam alinhados com $p_{IOI}$ — uma propriedade necessária para as médias condicionadas por template nas próximas etapas.

In [2]:
model = load_gpt2_small(device)
dataset = build_paired_dataset(
    model.tokenizer,
    n=N_DEV,
    seed=SEED,
    pattern=PATTERN,
)

print("pIOI:", tuple(dataset.ioi_tokens.shape))
print("pABC:", tuple(dataset.abc_tokens.shape))
print("templates cobertos:", len(dataset.groups), "/ 15")
print("prepend_bos:", dataset.prepend_bos)
assert dataset.prepend_bos is False
assert torch.equal(dataset.ioi_attention_mask, dataset.abc_attention_mask)
assert len(dataset.groups) == 15

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

pIOI: (256, 21)
pABC: (256, 21)
templates cobertos: 15 / 15
prepend_bos: False


In [3]:
rows = torch.arange(len(dataset))
pos = dataset.positions

assert torch.equal(dataset.ioi_tokens[rows, pos["IO"]], dataset.io_token_ids)
assert torch.equal(dataset.ioi_tokens[rows, pos["S1"]], dataset.s_token_ids)
assert torch.equal(dataset.ioi_tokens[rows, pos["S2"]], dataset.s_token_ids)
assert torch.equal(pos["S1+1"], pos["S1"] + 1)
assert torch.equal(dataset.ioi_tokens[rows, pos["END"] + 1], dataset.io_token_ids)
assert torch.equal(dataset.abc_tokens[rows, pos["S2"]], dataset.abc_token_ids)

for index in range(2):
    m = dataset.metadata[index]
    print(f"\nExemplo {index} ({m.pattern}, template {m.template_id})")
    print("pIOI:", dataset.ioi_prompts[index])
    print("pABC:", dataset.abc_prompts[index])
    print({name: int(values[index]) for name, values in pos.items()})

print("\nTokenização e posições semânticas verificadas.")


Exemplo 0 (BABA, template 10)
pIOI: After the lunch, Alicia and Matthew went to the school. Alicia gave a basketball to Matthew
pABC: After the lunch, Alicia and Matthew went to the school. Thomas gave a basketball to Matthew
{'IO': 6, 'S1': 4, 'S2': 12, 'S1+1': 5, 'END': 16}

Exemplo 1 (BABA, template 2)
pIOI: Then, Alicia and Alexander were working at the garden. Alicia decided to give a drink to Alexander
pABC: Then, Alicia and Alexander were working at the garden. Dustin decided to give a drink to Alexander
{'IO': 4, 'S1': 2, 'S2': 11, 'S1+1': 3, 'END': 17}

Tokenização e posições semânticas verificadas.


## Baseline no índice `END`

Os prompts têm comprimentos diferentes e recebem padding. Portanto, `logits[:, -1]` não representa o fim semântico de todos os exemplos. Cada métrica abaixo reúne os logits em `positions["END"]` por exemplo.

Para limitar VRAM, processamos o lote de desenvolvimento em minibatches e descartamos os logits completos após extrair as três métricas.

In [4]:
all_logit_differences = []
all_accuracies = []
all_io_probabilities = []

with torch.inference_mode():
    for start in range(0, len(dataset), BATCH_SIZE):
        stop = min(start + BATCH_SIZE, len(dataset))
        logits = model(dataset.ioi_tokens[start:stop].to(device))
        end = pos["END"][start:stop].to(device)
        io_ids = dataset.io_token_ids[start:stop].to(device)
        s_ids = dataset.s_token_ids[start:stop].to(device)
        all_logit_differences.append(
            logit_difference(
                logits, io_ids, s_ids, end, reduction="none"
            ).cpu()
        )
        all_accuracies.append(
            io_accuracy(
                logits, io_ids, s_ids, end, reduction="none"
            ).cpu()
        )
        all_io_probabilities.append(
            io_probability(logits, io_ids, end, reduction="none").cpu()
        )

observed = {
    "mean_logit_difference": torch.cat(all_logit_differences).mean().item(),
    "io_accuracy": torch.cat(all_accuracies).mean().item(),
    "mean_io_probability": torch.cat(all_io_probabilities).mean().item(),
}
assert observed["mean_logit_difference"] > 0
assert 0 <= observed["io_accuracy"] <= 1
assert 0 <= observed["mean_io_probability"] <= 1
observed

{'mean_logit_difference': 3.6294949054718018,
 'io_accuracy': 0.99609375,
 'mean_io_probability': 0.5307787656784058}

In [5]:
labels = {
    "mean_logit_difference": "Diferença média de logits IO−S",
    "io_accuracy": "Acurácia IO>S",
    "mean_io_probability": "Probabilidade média de IO",
}
for metric, target in PAPER_TARGETS.items():
    value = observed[metric]
    suffix = "%" if metric != "mean_logit_difference" else ""
    scale = 100 if suffix else 1
    print(
        f"{labels[metric]}: observado {value * scale:.3f}{suffix} | "
        f"paper {target * scale:.3f}{suffix} | "
        f"desvio {(value - target) * scale:+.3f}{suffix}"
    )

Diferença média de logits IO−S: observado 3.629 | paper 3.560 | desvio +0.069
Acurácia IO>S: observado 99.609% | paper 99.300% | desvio +0.309%
Probabilidade média de IO: observado 53.078% | paper 49.000% | desvio +4.078%


## Interpretação e limite amostral

O artigo reporta 3,56 de diferença média de logits, 99,3% de acurácia e 49% de probabilidade média de IO em **100.000 exemplos**. Aqui usamos apenas 256 exemplos para manter a execução rápida e a memória previsível. O lote menor, a seed fixa, a mistura específica de ABBA/BABA e diferenças numéricas da versão do modelo podem produzir desvios legítimos.

Por isso, a comparação é informativa: verificamos direção, ordem de grandeza e desempenho alto, mas não fazemos asserts contra os três números publicados.

In [6]:
result = {
    "paper": "arXiv:2211.00593",
    "seed": SEED,
    "config": {
        "n_examples": N_DEV,
        "paper_n_examples": 100_000,
        "batch_size": BATCH_SIZE,
        "pattern": PATTERN,
        "prepend_bos": dataset.prepend_bos,
        "device": str(device),
        "torch_version": torch.__version__,
        "transformer_lens_version": version("transformer-lens"),
    },
    "paper_targets": PAPER_TARGETS,
    "observed": observed,
    "deviation_from_paper": {
        metric: observed[metric] - target
        for metric, target in PAPER_TARGETS.items()
    },
}

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
output_path = project_root / "results/paper" / f"baseline_seed_{SEED}_n_{N_DEV}.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(
    json.dumps(result, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("Resultado salvo em", output_path.relative_to(project_root))
result

Resultado salvo em results\paper\baseline_seed_42_n_256.json


{'paper': 'arXiv:2211.00593',
 'seed': 42,
 'config': {'n_examples': 256,
  'paper_n_examples': 100000,
  'batch_size': 32,
  'pattern': 'mixed',
  'prepend_bos': False,
  'device': 'cuda',
  'torch_version': '2.15.0+cu132',
  'transformer_lens_version': '4.3.0'},
 'paper_targets': {'mean_logit_difference': 3.56,
  'io_accuracy': 0.993,
  'mean_io_probability': 0.49},
 'observed': {'mean_logit_difference': 3.6294949054718018,
  'io_accuracy': 0.99609375,
  'mean_io_probability': 0.5307787656784058},
 'deviation_from_paper': {'mean_logit_difference': 0.0694949054718017,
  'io_accuracy': 0.0030937500000000062,
  'mean_io_probability': 0.04077876567840577}}